# UK 5B Token Dataset Builder — Single Parquet Output

This notebook builds a UK training dataset from:

```text
/processed/UK
```

It writes **one output parquet file**:

```text
outputs/uk_5B_single/uk_5B_single.parquet
```

Design choices:

- The input directory may contain one or many parquet files, usually under `/processed/UK` on the server.
- The output is intentionally a **single parquet file**, as requested.
- The target token budget remains `5_000_000_000` unless you change `TARGET_TOKENS`.
- If the input directory contains fewer than 5B tokens, the notebook writes all available valid rows and reports the shortfall.

## Cell 0 — Optional package installation

**Why this cell exists**

The notebook requires `pyarrow` to read and write parquet files. Some JupyterHub kernels do not include it by default.

**Method**

Use the current notebook kernel's Python executable to install the required packages into the active environment.

**When to run**

Run this only if `import pyarrow` or `import pandas` fails.

In [ ]:
# Optional installation cell.
# Run only if imports fail.

# import sys
# !{sys.executable} -m pip install pyarrow pandas tqdm

## Cell 1 — Imports and configuration

**Purpose**

This cell imports the required libraries and defines the main dataset parameters: input path, output path, target token count, column names, and deduplication options.

**Method**

- `pathlib.Path` is used for stable path handling.
- `pyarrow.parquet` is used for streaming parquet reads and parquet writing.
- `pandas` is used for batch-level dataframe operations.
- `hashlib` is used to create stable exact-duplicate hashes.

**Why this design**

Large web corpora cannot safely be loaded into memory at once. The notebook therefore processes parquet files in batches.

In [ ]:
from pathlib import Path
import os
import json
import time
import random
import hashlib

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm

# -----------------------------
# Paths
# -----------------------------
# The notebook may be run from /home/jovyan/notebook.
# This resolver supports common UK dataset locations.
# Prefer /processed/UK if your server stores the processed UK parquet files there.
CANDIDATE_DATA_ROOTS = [
    Path("/processed/UK"),
    Path("/home/jovyan/processed/UK"),
    Path("/home/jovyan/data/UK"),
    Path("/home/jovyan/notebook/data/UK"),
    Path("data/UK"),
    Path("../data/UK"),
]

DATA_ROOT = None
for candidate in CANDIDATE_DATA_ROOTS:
    if candidate.exists():
        DATA_ROOT = candidate.resolve()
        break

if DATA_ROOT is None:
    DATA_ROOT = Path("/processed/UK").resolve()

OUTPUT_ROOT = Path("/home/jovyan/notebook/outputs/uk_5B_single")
OUTPUT_FILE = OUTPUT_ROOT / "uk_5B_single.parquet"
INDEX_FILE = OUTPUT_ROOT / "input_index.csv"
STATS_FILE = OUTPUT_ROOT / "sampling_stats.json"
README_FILE = OUTPUT_ROOT / "README.md"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Target
# -----------------------------
TARGET_TOKENS = 5_000_000_000
RANDOM_SEED = 8715

# -----------------------------
# Column names
# -----------------------------
TEXT_COL = "text"
TOKEN_COL = "token_count"
URL_COL = "url"

# If token_count is absent, approximate by whitespace.
APPROX_TOKEN_IF_MISSING = True

# Optional dedup columns.
ADD_HASH = True
ADD_SIMHASH = False  # Expensive at large scale; keep False unless explicitly needed.

HASH_COL = "hash"
SIMHASH_COL = "simhash"

# Batch size for streaming parquet.
MAX_ROWS_PER_READ_GROUP = 200_000

# Single-output setting.
# Because only one parquet is requested, this is not used for sharding.
WRITE_SINGLE_PARQUET = True

random.seed(RANDOM_SEED)

print("Current working directory:", Path.cwd())
print("DATA_ROOT:", DATA_ROOT)
print("DATA_ROOT exists:", DATA_ROOT.exists())
print("OUTPUT_FILE:", OUTPUT_FILE)

## Cell 2 — Locate input parquet files

**Purpose**

This cell finds all parquet files inside the UK dataset directory.

**Method**

It recursively searches for `*.parquet` files under `DATA_ROOT`.

**Why this design**

Some datasets store parquet files directly inside the batch folder, while others use nested subfolders. Recursive search handles both layouts.

In [ ]:
input_files = sorted(DATA_ROOT.rglob("*.parquet"))

print(f"Found {len(input_files)} parquet files under {DATA_ROOT}")

if len(input_files) == 0:
    print("\nNo parquet files found. Check these paths:")
    for p in CANDIDATE_DATA_ROOTS:
        print(f"- {p.resolve()} | exists={p.exists()}")

assert len(input_files) > 0, f"No parquet files found under {DATA_ROOT}"

print("\nFirst files:")
for p in input_files[:10]:
    print("-", p)

if len(input_files) > 10:
    print("...")
    for p in input_files[-5:]:
        print("-", p)

## Cell 3 — Build a lightweight parquet index

**Purpose**

Before writing the final dataset, this cell records basic metadata for each source parquet file.

**Method**

For each file, it reads parquet metadata such as row count, file size, schema columns, and token totals if `token_count` exists.

**Why this design**

The index lets you confirm whether the source data has enough tokens before doing the heavier full write. Reading only `token_count` is much cheaper than reading full text.

In [ ]:
def estimate_tokens_from_file(path: Path, token_col: str = TOKEN_COL):
    pf = pq.ParquetFile(path)
    cols = pf.schema.names
    row_count = pf.metadata.num_rows
    size_bytes = path.stat().st_size

    token_sum = None
    has_token_count = token_col in cols

    if has_token_count:
        token_sum = 0
        for batch in pf.iter_batches(columns=[token_col], batch_size=MAX_ROWS_PER_READ_GROUP):
            s = batch.column(0).to_pandas()
            token_sum += int(pd.to_numeric(s, errors="coerce").fillna(0).sum())

    return {
        "file_path": str(path),
        "file_name": path.name,
        "rows": row_count,
        "size_bytes": size_bytes,
        "has_text": TEXT_COL in cols,
        "has_token_count": has_token_count,
        "token_sum": token_sum,
        "columns": ",".join(cols),
    }


index_rows = []
for p in tqdm(input_files, desc="Indexing parquet files"):
    try:
        index_rows.append(estimate_tokens_from_file(p))
    except Exception as e:
        index_rows.append({
            "file_path": str(p),
            "file_name": p.name,
            "rows": None,
            "size_bytes": p.stat().st_size if p.exists() else None,
            "has_text": False,
            "has_token_count": False,
            "token_sum": None,
            "columns": "",
            "error": repr(e),
        })

index_df = pd.DataFrame(index_rows)
index_df.to_csv(INDEX_FILE, index=False)

display(index_df.head())
print("Index saved to:", INDEX_FILE)
print("Total rows:", int(index_df["rows"].fillna(0).sum()))

if index_df["token_sum"].notna().any():
    print("Known total tokens:", f'{int(index_df["token_sum"].dropna().sum()):,}')
else:
    print("Known total tokens: unknown because token_count column was not available")

## Cell 4 — Validate token availability

**Purpose**

This cell checks whether the source directory can realistically satisfy the 5B-token target.

**Method**

It compares the sum of `token_count` from the index against `TARGET_TOKENS`.

**Why this design**

If the input folder is only one Common Crawl batch, it may contain fewer than 5B tokens. The notebook will still generate a valid single parquet file, but this check tells you whether it reached the requested budget.

In [ ]:
known_total_tokens = index_df["token_sum"].dropna().sum() if index_df["token_sum"].notna().any() else None

print(f"Target tokens: {TARGET_TOKENS:,}")

if known_total_tokens is not None:
    known_total_tokens = int(known_total_tokens)
    print(f"Known total tokens in input: {known_total_tokens:,}")

    if known_total_tokens < TARGET_TOKENS:
        print("WARNING: input appears below target. The output will contain all available valid rows unless approximate token counting changes the estimate.")
    else:
        print("OK: input appears sufficient for the target.")
else:
    print("token_count is not available. Token counts will be approximated from text during writing.")

## Cell 5 — Hash and token-count helper functions

**Purpose**

This cell defines reusable functions for token counting and deduplication metadata.

**Method**

- If `token_count` is missing, approximate token count by whitespace splitting.
- If `ADD_HASH=True`, add a SHA-1 hash of the `text` column.
- If `ADD_SIMHASH=True`, add a simple 64-bit SimHash for near-duplicate detection.

**Why this design**

Exact hash is useful for exact duplicate detection and is relatively cheap. SimHash is useful for near duplicates, but it is expensive for very large corpora, so it is disabled by default.

In [ ]:
def stable_text_hash(text: str) -> str:
    if text is None:
        text = ""
    if not isinstance(text, str):
        text = str(text)
    return hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()


def simhash_64(text: str, ngram: int = 3) -> int:
    if text is None:
        text = ""
    if not isinstance(text, str):
        text = str(text)

    text = " ".join(text.lower().split())

    if len(text) < ngram:
        grams = [text]
    else:
        grams = [text[i:i + ngram] for i in range(len(text) - ngram + 1)]

    v = [0] * 64
    for g in grams:
        h = int(hashlib.blake2b(g.encode("utf-8", errors="ignore"), digest_size=8).hexdigest(), 16)
        for i in range(64):
            if h & (1 << i):
                v[i] += 1
            else:
                v[i] -= 1

    out = 0
    for i, x in enumerate(v):
        if x >= 0:
            out |= (1 << i)
    return out


def add_token_count_if_needed(df: pd.DataFrame) -> pd.DataFrame:
    if TOKEN_COL not in df.columns:
        if not APPROX_TOKEN_IF_MISSING:
            raise ValueError(f"Missing {TOKEN_COL} and APPROX_TOKEN_IF_MISSING=False")
        df[TOKEN_COL] = df[TEXT_COL].fillna("").astype(str).str.split().str.len().astype("int64")
    else:
        df[TOKEN_COL] = pd.to_numeric(df[TOKEN_COL], errors="coerce").fillna(0).astype("int64")
    return df


def add_dedup_columns(df: pd.DataFrame) -> pd.DataFrame:
    if ADD_HASH:
        df[HASH_COL] = df[TEXT_COL].map(stable_text_hash)
    if ADD_SIMHASH:
        df[SIMHASH_COL] = df[TEXT_COL].map(simhash_64).astype("uint64")
    return df

## Cell 6 — Stream source parquet files into one output parquet

**Purpose**

This is the main construction cell. It reads the source parquet files in batches and writes the selected rows into one final parquet file.

**Method**

- Open a single `ParquetWriter`.
- Read each source file by batch.
- Keep rows with positive token counts.
- Add hash columns if configured.
- Stop once `TARGET_TOKENS` is reached.
- Close the writer at the end.

**Why this design**

Using `ParquetWriter` allows a single output parquet file without holding the full 5B-token dataset in memory.

In [ ]:
start_time = time.time()

if OUTPUT_FILE.exists():
    print("Existing output file found and will be overwritten:", OUTPUT_FILE)
    OUTPUT_FILE.unlink()

total_tokens_written = 0
total_rows_written = 0
per_input_stats = []

writer = None
output_schema = None

try:
    for path in tqdm(input_files, desc="Writing single parquet"):
        if total_tokens_written >= TARGET_TOKENS:
            break

        file_rows_written = 0
        file_tokens_written = 0

        pf = pq.ParquetFile(path)
        cols = pf.schema.names

        if TEXT_COL not in cols:
            print(f"Skipping {path}: no {TEXT_COL} column")
            continue

        read_cols = [TEXT_COL]
        for c in [TOKEN_COL, URL_COL]:
            if c in cols and c not in read_cols:
                read_cols.append(c)

        for batch in pf.iter_batches(columns=read_cols, batch_size=MAX_ROWS_PER_READ_GROUP):
            if total_tokens_written >= TARGET_TOKENS:
                break

            df = batch.to_pandas()
            df = add_token_count_if_needed(df)
            df = df[df[TOKEN_COL] > 0].copy()

            if len(df) == 0:
                continue

            df = add_dedup_columns(df)

            remaining = TARGET_TOKENS - total_tokens_written
            cumsum = df[TOKEN_COL].cumsum()
            keep_mask = cumsum <= remaining

            if not keep_mask.all():
                if keep_mask.sum() == 0:
                    # Include one row if the next row alone crosses the target.
                    df = df.iloc[:1].copy()
                else:
                    df = df.loc[keep_mask].copy()

            batch_tokens = int(df[TOKEN_COL].sum())
            batch_rows = len(df)

            if batch_rows == 0:
                continue

            table = pa.Table.from_pandas(df, preserve_index=False)

            if writer is None:
                output_schema = table.schema
                writer = pq.ParquetWriter(
                    OUTPUT_FILE,
                    output_schema,
                    compression="zstd",
                    use_dictionary=True,
                    write_statistics=True,
                )
            else:
                # Keep schema stable across batches.
                table = table.cast(output_schema)

            writer.write_table(table)

            total_tokens_written += batch_tokens
            total_rows_written += batch_rows
            file_rows_written += batch_rows
            file_tokens_written += batch_tokens

        per_input_stats.append({
            "input_file": str(path),
            "rows_written": file_rows_written,
            "tokens_written": file_tokens_written,
        })

finally:
    if writer is not None:
        writer.close()

elapsed = time.time() - start_time

summary = {
    "dataset": "UK 5B Token Dataset - Single Parquet",
    "country": "UK",
    "input_dir": str(DATA_ROOT),
    "output_file": str(OUTPUT_FILE),
    "target_tokens": TARGET_TOKENS,
    "actual_tokens_written": total_tokens_written,
    "actual_rows_written": total_rows_written,
    "num_input_files_found": len(input_files),
    "single_output_parquet": True,
    "random_seed": RANDOM_SEED,
    "add_hash": ADD_HASH,
    "add_simhash": ADD_SIMHASH,
    "elapsed_seconds": elapsed,
    "per_input_stats": per_input_stats,
}

with open(STATS_FILE, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\nDONE")
print(f"Output file:            {OUTPUT_FILE}")
print(f"Actual tokens written:  {total_tokens_written:,}")
print(f"Actual rows written:    {total_rows_written:,}")
print(f"Elapsed seconds:        {elapsed:.1f}")
print(f"Stats saved to:         {STATS_FILE}")

## Cell 7 — Verify the single output parquet

**Purpose**

This cell validates that the final parquet file exists and that its row/token counts match the writing summary.

**Method**

It reopens the output parquet file, reads only `token_count`, and sums it.

**Why this design**

Verification should be independent of the writing counters. Re-reading the saved file confirms that the parquet file is readable and that token counts were written correctly.

In [ ]:
assert OUTPUT_FILE.exists(), f"Output file does not exist: {OUTPUT_FILE}"

pf = pq.ParquetFile(OUTPUT_FILE)
verify_rows = pf.metadata.num_rows
verify_tokens = 0

if TOKEN_COL in pf.schema.names:
    for batch in tqdm(pf.iter_batches(columns=[TOKEN_COL], batch_size=MAX_ROWS_PER_READ_GROUP), desc="Verifying token_count"):
        s = batch.column(0).to_pandas()
        verify_tokens += int(pd.to_numeric(s, errors="coerce").fillna(0).sum())
else:
    print(f"WARNING: {TOKEN_COL} not found in output schema")

print(f"Output file:      {OUTPUT_FILE}")
print(f"Verified rows:    {verify_rows:,}")
print(f"Verified tokens:  {verify_tokens:,}")
print(f"Target tokens:    {TARGET_TOKENS:,}")

if verify_tokens >= TARGET_TOKENS:
    print("OK: output reaches or exceeds target.")
else:
    print("WARNING: output is below target. This likely means the input directory contains fewer than the requested tokens.")

## Cell 8 — Create a Hugging Face style README

**Purpose**

This cell creates a simple dataset card for the generated UK dataset.

**Method**

It writes a Markdown file with dataset summary, source path, token count, row count, and output location.

**Why this design**

A README makes the dataset easier to upload, review, and explain to teammates or stakeholders.

In [ ]:
readme = f'''---
license: other
language:
- en
task_categories:
- text-generation
pretty_name: UK 5B Token Dataset Single Parquet
size_categories:
- 1B<n<10B
---

# UK 5B Token Dataset Single Parquet

## Dataset Summary

This dataset is a UK web-corpus training dataset prepared from a Common Crawl batch.

## Source Data

Source directory:

```text
{DATA_ROOT}
```

## Output

Single output parquet:

```text
{OUTPUT_FILE}
```

## Construction Method

The dataset was created by streaming source parquet files in chunks and writing rows to a single parquet file until the target token budget was reached or the source data was exhausted.

- Country: UK
- Source directory: UK processed parquet data
- Target tokens requested: {TARGET_TOKENS:,}
- Actual tokens written: {verify_tokens:,}
- Actual rows written: {verify_rows:,}
- Single parquet output: True
- Random seed: {RANDOM_SEED}
- Exact hash column: {ADD_HASH}
- SimHash column: {ADD_SIMHASH}

## Columns

Expected core columns:

- `text`
- `token_count`
- `url` if available
- `hash` if exact hashing was enabled
- `simhash` if near-duplicate hashing was enabled

## Intended Use

This dataset is intended for UK language model training experiments.
'''

README_FILE.write_text(readme, encoding="utf-8")
print("README saved to:", README_FILE)

## Cell 9 — DDP / multi-GPU training note

**Purpose**

This cell explains where DDP fits in the workflow.

**Method**

Dataset construction is completed first. DDP is then used in the model training script to distribute batches across GPUs.

**Why this design**

DDP is designed for parallel model training. It is not the best tool for parquet file construction. For data construction, streaming parquet I/O is more reliable and easier to debug.

In [ ]:
print("Dataset construction output:")
print(OUTPUT_FILE)

print("\nExample training command after you prepare a model training script:")
print("torchrun --nproc_per_node=4 train_uk_ddp.py --data_file", OUTPUT_FILE)